# Tutorial 10 Autoencoders and Variational Autoencoders

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JimengShi/SysEng_Cornell/blob/main/Tutorial_10_Generative_Modeling_VAE/tutorial_10_pytorch.ipynb)

Learn reconstruction, denoising, anomaly scoring, and latent-variable generation with PyTorch. Compare deterministic autoencoders, a convolutional VAE, and a small adversarial autoencoder.

**Start:** Open in Colab → connect → **Runtime → Run all**, or use Shift+Enter to work cell by cell. Required package setup is included below. No local installation, API key, or Drive mount is needed.

**Prerequisites:** Tutorials 03–05; tensors, neural networks, training/validation/test splits.

**Resources:** CPU-compatible quick mode; Fashion-MNIST, MNIST, and ECG downloads (~50 MB combined). Five small models are trained, including a two-epoch adversarial demonstration. Allow 90–120 minutes for reading and exercises.

Each notebook is independent: earlier lessons are helpful but their runtime state and files are not required. While the GitHub repository is private, authorize Colab to access it, or download this notebook and use **File → Upload notebook** in Colab.

## Learning goals

1. Distinguish reconstruction of an input from generation using a prior sample.
2. Train a denoiser with noisy inputs and clean targets; compare with leaving the noise unchanged.
3. Calibrate an anomaly threshold without using test labels and state which class is positive.
4. Implement Gaussian reparameterization and analytic KL divergence; track reconstruction and KL in consistent units.
5. Inspect posterior means, prior samples, interpolation, and a two-dimensional latent grid.
6. Verify that adversarial training updates the **same encoder** used by the decoder, with correct gradient isolation.

A deterministic AE does not define a useful sampling distribution automatically. A VAE regularizes approximate posteriors toward a prior, but neither realistic samples nor calibrated uncertainty are guaranteed. The 2D bottleneck makes visualization easy at the cost of capacity. These are instructional experiments, not generation-quality benchmarks. The ECG section illustrates reconstruction-error classification on a public dataset; it is not a clinical diagnostic system.

## 1. Environment

This cell installs missing packages inside the current runtime. It keeps an existing PyTorch/torchvision pair together and does not downgrade Colab’s installed stack. If imports fail after other experiments, start a fresh runtime and run from the top.

In [ ]:
import importlib.util
import subprocess
import sys

packages = {'torch': 'torch', 'torchvision': 'torchvision', 'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn', 'PIL': 'pillow'}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
if "torch" in missing or "torchvision" in missing:
    missing = sorted(set(missing + ["torch", "torchvision"]))
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
else:
    print("Required packages are already installed.")

In [ ]:
import json, math, time, random, platform, hashlib, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,TensorDataset
import torchvision
from torchvision import datasets
from torchvision.utils import make_grid
from PIL import Image
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score
from IPython.display import display, Image as DisplayImage
%matplotlib inline
SEED=42
QUICK_RUN=True
DEVICE_PREFERENCE="auto"
RUN_AAE=True  # Small advanced demonstration; set False to skip.
assert DEVICE_PREFERENCE in {"auto","cpu"}
device=torch.device("cuda" if DEVICE_PREFERENCE=="auto" and torch.cuda.is_available() else "cpu")
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(2,torch.get_num_threads()))
DATA_DIR=Path("data/tutorial_10");OUTPUT_DIR=Path("outputs/tutorial_10")
DATA_DIR.mkdir(parents=True,exist_ok=True);OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
versions={"python":platform.python_version(),"torch":str(torch.__version__),"torchvision":str(torchvision.__version__),
    "numpy":np.__version__,"pandas":pd.__version__,"matplotlib":matplotlib.__version__,"sklearn":sklearn.__version__,
    "device":str(device),"seed":SEED,"quick_run":QUICK_RUN,"run_aae":RUN_AAE}
display(pd.Series(versions,name="Environment"))
run_started=time.perf_counter()
figures=[]
def save_figure(fig,name):
    fig.savefig(OUTPUT_DIR/name,dpi=140,bbox_inches="tight");figures.append(name);plt.show()
def image_rows(tensors,labels,name):
    columns=min(8,min(len(x) for x in tensors))
    fig,axes=plt.subplots(len(tensors),columns,figsize=(12,1.8*len(tensors)),squeeze=False,layout="constrained")
    for row,(images,label) in enumerate(zip(tensors,labels)):
        for col in range(columns):
            axes[row,col].imshow(images[col].detach().cpu().squeeze(),cmap="gray",vmin=0,vmax=1)
            axes[row,col].set_xticks([]);axes[row,col].set_yticks([])
        axes[row,0].set_ylabel(label)
    save_figure(fig,name)
def loader(x,y=None,shuffle=False,seed=SEED,batch=128):
    return DataLoader(TensorDataset(x,x if y is None else y),batch_size=batch,shuffle=shuffle,num_workers=0,
                      generator=torch.Generator().manual_seed(seed))

## 2. Prepare image data once

Fashion-MNIST supports reconstruction and denoising; MNIST supports latent-variable generation. Use balanced subsets in quick mode. Train and validation indices come from the original training partition; test images remain in the original test partition. Labels only balance splits and color a later latent plot; the reconstruction objectives do not use them.

Fashion-MNIST stays grayscale in [0,1]. MNIST is **statically binarized at 0.5** so the VAE's Bernoulli likelihood has binary observations. This differs from dynamic-binarization benchmarks. Quick mode downloads the complete source archives even though training uses subsets.

In [ ]:
FASHION_REVISION="b2617bb6d3ffa2e429640350f613e3291e10b141"
class HTTPSFashionMNIST(datasets.FashionMNIST):
    mirrors=[f"https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/{FASHION_REVISION}/data/fashion/"]
def stratified_indices(labels,counts,seed):
    rng=np.random.default_rng(seed);groups=[[] for _ in counts]
    for label in range(10):
        candidates=rng.permutation(np.flatnonzero(np.asarray(labels)==label));assert sum(counts)<=len(candidates)
        start=0
        for group,count in zip(groups,counts):group.extend(candidates[start:start+count]);start+=count
    return [rng.permutation(group) for group in groups]
def image_splits(dataset_class,counts,binary=False):
    train=dataset_class(root=str(DATA_DIR),train=True,download=True)
    test=dataset_class(root=str(DATA_DIR),train=False,download=True)
    ti,vi=stratified_indices(train.targets.numpy(),counts[:2],SEED)
    (si,)=stratified_indices(test.targets.numpy(),counts[2:],SEED+1)
    assert set(ti).isdisjoint(vi)
    tensors=[]
    for source,indices in [(train,ti),(train,vi),(test,si)]:
        x=source.data[indices].unsqueeze(1).float()/255
        tensors.append((x>0.5).float() if binary else x)
    return (*tensors,test.targets[si])
f_train,f_val,f_test,f_labels=image_splits(HTTPSFashionMNIST,(400,80,100) if QUICK_RUN else (5000,1000,1000))
m_train,m_val,m_test,m_labels=image_splits(datasets.MNIST,(600,100,100) if QUICK_RUN else (5000,400,80),binary=True)
# Larger MNIST counts are limited by the smallest class; they are not the full source set.
print("Fashion train/val/test:",[len(x) for x in (f_train,f_val,f_test)])
print("MNIST train/val/test:",[len(x) for x in (m_train,m_val,m_test)])
assert f_train.shape[1:]==(1,28,28) and set(m_train.unique().tolist())=={0.0,1.0}
image_rows([f_train[:8],m_train[:8]],["Fashion (gray)","MNIST (binary)"],"data_examples.png")

## 3. Deterministic reconstruction and denoising

A dense AE compresses 784 pixels into 32 numbers. A convolutional denoiser retains a spatial code. Both minimize mean squared error **per pixel**; they have different architectures, so this is not a controlled architecture benchmark.

Fresh Gaussian noise with standard deviation 0.3 is added to training images each epoch and clipped to [0,1]. Validation and test corruptions use separate fixed seeds. Select each model on validation MSE, restore its best epoch, then evaluate test data once. Training-mean-image reconstruction and unchanged noisy input are simple baselines.

In [ ]:
class DenseAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Flatten(),nn.Linear(784,128),nn.ReLU(),nn.Linear(128,32))
        self.decoder=nn.Sequential(nn.Linear(32,128),nn.ReLU(),nn.Linear(128,784),nn.Sigmoid(),nn.Unflatten(1,(1,28,28)))
    def forward(self,x):return self.decoder(self.encoder(x))
class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Conv2d(1,8,3,2,1),nn.ReLU(),nn.Conv2d(8,16,3,2,1),nn.ReLU())
        self.decoder=nn.Sequential(nn.ConvTranspose2d(16,8,4,2,1),nn.ReLU(),nn.ConvTranspose2d(8,1,4,2,1),nn.Sigmoid())
    def forward(self,x):return self.decoder(self.encoder(x))
def corrupt(x,seed):
    noise=torch.randn(x.shape,generator=torch.Generator().manual_seed(seed))
    return (x+0.3*noise).clamp(0,1)
noisy_val=corrupt(f_val,SEED+100);noisy_test=corrupt(f_test,SEED+101)
def reconstruction_pass(network,data,optimizer=None,mae=False):
    network.train(optimizer is not None);total=count=0
    with torch.set_grad_enabled(optimizer is not None):
        for x,target in data:
            x,target=x.to(device),target.to(device);pred=network(x)
            assert pred.shape==target.shape
            per_example=(pred-target).abs().flatten(1).mean(1) if mae else (pred-target).square().flatten(1).mean(1)
            loss=per_example.mean();assert torch.isfinite(loss)
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
            total+=per_example.sum().item();count+=len(x)
    return total/count
def fit_reconstruction(network,train,validation,epochs,noise=False,mae=False):
    network.to(device);optimizer=torch.optim.Adam(network.parameters(),lr=0.001)
    best=float("inf");best_epoch=0;history=[]
    for epoch in range(1,epochs+1):
        inputs=corrupt(train,SEED+epoch) if noise else train
        training=reconstruction_pass(network,loader(inputs,train,True,SEED+epoch),optimizer,mae)
        valid=reconstruction_pass(network,validation,mae=mae)
        history.append({"epoch":epoch,"train":training,"validation":valid})
        if valid<best:
            best=valid;best_epoch=epoch;state={k:v.detach().cpu().clone() for k,v in network.state_dict().items()}
        if epochs<=10 or epoch%10==0:print(f"{type(network).__name__} epoch {epoch}: train {training:.5f}, validation {valid:.5f}")
    network.load_state_dict(state);network.eval()
    return pd.DataFrame(history),best_epoch
@torch.no_grad()
def reconstruct(network,x):
    network.eval();return torch.cat([network(batch.to(device)).cpu() for batch,_ in loader(x)])
ae=DenseAE();denoiser=Denoiser()
ae_history,ae_best=fit_reconstruction(ae,f_train,loader(f_val),4 if QUICK_RUN else 15)
dn_history,dn_best=fit_reconstruction(denoiser,f_train,loader(noisy_val,f_val),4 if QUICK_RUN else 15,noise=True)

In [ ]:
ae_reconstruction=reconstruct(ae,f_test);denoised=reconstruct(denoiser,noisy_test)
image_metrics={"AE_test_MSE":float((ae_reconstruction-f_test).square().mean()),
    "training_mean_image_test_MSE":float((f_train.mean(0,keepdim=True)-f_test).square().mean()),
    "denoiser_test_MSE":float((denoised-f_test).square().mean()),"unchanged_noisy_test_MSE":float((noisy_test-f_test).square().mean())}
print(json.dumps(image_metrics,indent=2))
image_rows([f_test,ae_reconstruction,noisy_test,denoised],["Clean","Dense AE","Noisy","Denoised"],"reconstruction_denoising.png")
fig,axes=plt.subplots(1,2,figsize=(10,3.3),layout="constrained")
for ax,history,title in zip(axes,[ae_history,dn_history],["Dense AE","Convolutional denoiser"]):
    ax.plot(history.epoch,history.train,label="Training");ax.plot(history.epoch,history.validation,label="Validation")
    ax.set(title=title,xlabel="Epoch",ylabel="MSE per pixel");ax.legend()
save_figure(fig,"reconstruction_curves.png")

## 4. ECG reconstruction-error classification

The source lesson's public CSV has 140 features and labels **1=normal, 0=abnormal**. Our positive prediction means **anomaly**, so evaluation uses `anomaly = (label == 0)`. Reversing this convention reverses the interpretation of precision and recall.

Remove exact duplicate feature rows, then make stratified 60/20/20 train/validation/test partitions. Normalize using the normal training subset only. Fit and select the AE using normal training/validation examples; abnormal examples are not required for fitting. Fix the threshold at the **95th percentile of normal validation MAE**, never the test distribution. This sets an empirical calibration target, not a guaranteed future false-positive rate. No patient identifiers are provided here, so this is a row-level split and cannot establish patient-independent performance. Near-duplicates and clinical utility are not assessed.

In [ ]:
ECG_URL="https://storage.googleapis.com/download.tensorflow.org/data/ecg.csv"
ecg_path=DATA_DIR/"ecg.csv"
if not ecg_path.exists():
    partial=ecg_path.with_suffix(".part")
    with urllib.request.urlopen(ECG_URL,timeout=120) as response:partial.write_bytes(response.read())
    partial.replace(ecg_path)
ecg_sha=hashlib.sha256(ecg_path.read_bytes()).hexdigest()
df=pd.read_csv(ecg_path,header=None)
assert df.shape[1]==141 and np.isfinite(df.to_numpy()).all()
assert set(df[140].unique())=={0.0,1.0}
# Identical feature rows must not carry contradictory labels.
assert df.groupby(list(range(140)),dropna=False)[140].nunique().max()==1
before=len(df);df=df.drop_duplicates(subset=list(range(140))).reset_index(drop=True)
values=df.iloc[:,:140].to_numpy(dtype=np.float32);normal=df[140].to_numpy()==1
indices=np.arange(len(df))
train_idx,remaining=train_test_split(indices,test_size=0.4,stratify=normal,random_state=SEED)
val_idx,test_idx=train_test_split(remaining,test_size=0.5,stratify=normal[remaining],random_state=SEED)
assert set(train_idx).isdisjoint(val_idx) and set(train_idx).isdisjoint(test_idx) and set(val_idx).isdisjoint(test_idx)
normal_training=values[train_idx][normal[train_idx]]
low=float(normal_training.min());scale=float(normal_training.max()-low);assert scale>0
scaled=torch.from_numpy((values-low)/scale)
ecg_train=scaled[train_idx][normal[train_idx]];ecg_val=scaled[val_idx][normal[val_idx]];ecg_test=scaled[test_idx]
y_anomaly=~normal[test_idx]
class ECGAutoencoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=nn.Sequential(nn.Linear(140,32),nn.ReLU(),nn.Linear(32,8))
        # Linear output allows held-out values outside the training normalization range.
        self.decoder=nn.Sequential(nn.Linear(8,32),nn.ReLU(),nn.Linear(32,140))
    def forward(self,x):return self.decoder(self.encoder(x))
ecg_model=ECGAutoencoder()
ecg_history,ecg_best=fit_reconstruction(ecg_model,ecg_train,loader(ecg_val),50 if QUICK_RUN else 150,mae=True)
calibration_errors=(reconstruct(ecg_model,ecg_val)-ecg_val).abs().mean(1).numpy()
threshold=float(np.quantile(calibration_errors,0.95))
print("Rows / exact duplicates removed:",len(df),before-len(df))
print("Train / validation / test:",[len(i) for i in (train_idx,val_idx,test_idx)])
print("Normal fitting / calibration rows:",len(ecg_train),len(ecg_val),"Threshold:",threshold)

In [ ]:
ecg_prediction=reconstruct(ecg_model,ecg_test)
test_errors=(ecg_prediction-ecg_test).abs().mean(1).numpy()
predicted_anomaly=test_errors>threshold
cm=confusion_matrix(y_anomaly,predicted_anomaly,labels=[False,True]);tn,fp,fn,tp=cm.ravel()
ecg_metrics={"positive_class":"anomaly (CSV label 0)","threshold":threshold,"threshold_rule":"95th percentile of normal validation MAE",
    "normal_validation_exceedance":float((calibration_errors>threshold).mean()),
    "accuracy":float((predicted_anomaly==y_anomaly).mean()),"precision":float(precision_score(y_anomaly,predicted_anomaly,zero_division=0)),
    "recall":float(recall_score(y_anomaly,predicted_anomaly,zero_division=0)),"f1":float(f1_score(y_anomaly,predicted_anomaly,zero_division=0)),
    "roc_auc":float(roc_auc_score(y_anomaly,test_errors)),"test_false_positive_rate":float(fp/(fp+tn)),
    "always_normal_accuracy":float((~y_anomaly).mean()),"confusion_matrix":cm.tolist()}
assert abs(ecg_metrics["recall"]-tp/(tp+fn))<1e-12
print(json.dumps(ecg_metrics,indent=2))
fig,axes=plt.subplots(1,3,figsize=(13,3.5),layout="constrained")
axes[0].plot(ecg_history.epoch,ecg_history.train,label="Normal training");axes[0].plot(ecg_history.epoch,ecg_history.validation,label="Normal validation")
axes[0].set(xlabel="Epoch",ylabel="MAE",title="Normal-only model selection");axes[0].legend(fontsize=8)
for flag,label in [(False,"Normal"),(True,"Anomaly")]:axes[1].hist(test_errors[y_anomaly==flag],bins=25,alpha=0.6,label=label)
axes[1].axvline(threshold,color="black",ls="--",label="Fixed validation threshold");axes[1].set(xlabel="Test reconstruction MAE",ylabel="Count");axes[1].legend(fontsize=8)
axes[2].imshow(cm,cmap="Blues");axes[2].set(xticks=[0,1],yticks=[0,1],xticklabels=["Normal","Anomaly"],yticklabels=["Normal","Anomaly"],xlabel="Predicted",ylabel="True",title="Anomaly is the positive class")
for i in range(2):
    for j in range(2):axes[2].text(j,i,str(cm[i,j]),ha="center",va="center",color="white" if cm[i,j]>cm.max()/2 else "black")
save_figure(fig,"ecg_evaluation.png")
fig,axes=plt.subplots(1,2,figsize=(10,3),layout="constrained")
for ax,flag in zip(axes,[False,True]):
    idx=int(np.flatnonzero(y_anomaly==flag)[0]);ax.plot(ecg_test[idx],label="Input");ax.plot(ecg_prediction[idx],label="Reconstruction")
    ax.set(title="First held-out anomaly" if flag else "First held-out normal",xlabel="Sample index",ylabel="Normalized value");ax.legend()
save_figure(fig,"ecg_examples.png")

## 5. VAE: distributions, not just codes

The encoder returns a diagonal Gaussian's mean `mu` and log variance `logvar`. Draw `epsilon ~ N(0,I)` and form `z = mu + exp(0.5*logvar) * epsilon`. Gradients flow through mu and logvar; epsilon does not need gradients. These parameters come from the **encoder**.

For each binary image, reconstruction cost is Bernoulli negative log-likelihood, implemented as BCE **with logits, summed across 784 pixels**. The analytic KL to N(0,I) is `0.5 * sum(mu² + exp(logvar) - 1 - logvar)`, summed across latent dimensions. Average the combined cost across images, not pixels. With beta=1, the expected objective is negative ELBO; one sampled z gives a Monte Carlo estimate and is not itself a guaranteed bound on each realization. Other beta values are weighted objectives, not the standard ELBO.

We use a 2D latent space to plot it directly. Validation uses fixed seeded epsilon so checkpoint comparisons do not change purely because of resampled noise. Test uses a separate seed. Decoding mu gives a deterministic visualization, **not** the expectation of the decoded distribution.

In [ ]:
LATENT_DIM=2;BETA=1.0
class ConvVAE(nn.Module):
    def __init__(self,latent_dim=2):
        super().__init__();self.latent_dim=latent_dim
        self.encoder=nn.Sequential(nn.Conv2d(1,8,3,2,1),nn.ReLU(),nn.Conv2d(8,16,3,2,1),nn.ReLU(),nn.Flatten(),nn.Linear(16*7*7,latent_dim*2))
        self.decoder=nn.Sequential(nn.Linear(latent_dim,16*7*7),nn.ReLU(),nn.Unflatten(1,(16,7,7)),
            nn.ConvTranspose2d(16,8,4,2,1),nn.ReLU(),nn.ConvTranspose2d(8,1,4,2,1))
    def encode(self,x):return self.encoder(x).chunk(2,dim=1)
    def decode(self,z):return self.decoder(z)  # Logits, not probabilities.
    def forward(self,x,eps=None):
        mu,logvar=self.encode(x)
        if eps is None:eps=torch.randn_like(mu)
        z=mu+torch.exp(0.5*logvar)*eps
        return self.decode(z),mu,logvar

def vae_components(logits,x,mu,logvar):
    recon=F.binary_cross_entropy_with_logits(logits,x,reduction="none").flatten(1).sum(1)
    kl=0.5*(mu.square()+logvar.exp()-1-logvar).sum(1)
    return recon,kl
# Analytic KL and pathwise-gradient sanity checks.
probe_mu=torch.tensor([[0.3,-0.2]],requires_grad=True);probe_logvar=torch.tensor([[0.1,-0.4]],requires_grad=True)
probe_eps=torch.tensor([[0.7,-0.6]])
probe_z=probe_mu+torch.exp(0.5*probe_logvar)*probe_eps
probe_z.sum().backward();assert probe_mu.grad.abs().sum()>0 and probe_logvar.grad.abs().sum()>0
q=torch.distributions.Normal(probe_mu.detach(),torch.exp(0.5*probe_logvar.detach()))
p=torch.distributions.Normal(torch.zeros_like(probe_mu),torch.ones_like(probe_mu))
formula=0.5*(probe_mu.detach().square()+probe_logvar.detach().exp()-1-probe_logvar.detach()).sum(1)
torch.testing.assert_close(formula,torch.distributions.kl_divergence(q,p).sum(1))
assert (0.5*(torch.zeros(2).square()+torch.zeros(2).exp()-1-torch.zeros(2))).sum()==0
vae=ConvVAE(LATENT_DIM).to(device)
with torch.no_grad():
    logits,mu,logvar=vae(m_train[:4].to(device),torch.zeros(4,LATENT_DIM,device=device))
    assert logits.shape==(4,1,28,28) and mu.shape==logvar.shape==(4,LATENT_DIM)
print("Reparameterization gradients, analytic KL, zero-KL, and shape checks passed.")

In [ ]:
def vae_pass(data,optimizer=None,seed=SEED+200):
    vae.train(optimizer is not None);totals=np.zeros(3);count=0
    evaluation_rng=torch.Generator().manual_seed(seed)
    with torch.set_grad_enabled(optimizer is not None):
        for x,_ in data:
            x=x.to(device)
            eps=None if optimizer is not None else torch.randn((len(x),LATENT_DIM),generator=evaluation_rng).to(device)
            logits,mu,logvar=vae(x,eps);recon,kl=vae_components(logits,x,mu,logvar)
            objective=recon+BETA*kl
            assert torch.isfinite(objective).all() and (kl>=-1e-6).all()
            if optimizer is not None:
                optimizer.zero_grad(set_to_none=True);objective.mean().backward()
                norm=nn.utils.clip_grad_norm_(vae.parameters(),10.0);assert torch.isfinite(norm)
                optimizer.step()
            totals+=np.array([objective.sum().item(),recon.sum().item(),kl.sum().item()]);count+=len(x)
    return dict(zip(["objective","reconstruction","KL"],(totals/count).tolist()))
fixed_prior=torch.randn(16,LATENT_DIM,generator=torch.Generator().manual_seed(SEED+300))
frames=[]
@torch.no_grad()
def prior_frame():
    vae.eval();prob=vae.decode(fixed_prior.to(device)).sigmoid().cpu()
    grid=make_grid(prob,nrow=4,padding=2).permute(1,2,0).numpy()
    frames.append(Image.fromarray((grid*255).round().astype(np.uint8)))
prior_frame()
vae_optimizer=torch.optim.Adam(vae.parameters(),lr=0.001)
vae_epochs=6 if QUICK_RUN else 20
vae_history=[];vae_best_loss=float("inf");vae_best_epoch=0
for epoch in range(1,vae_epochs+1):
    training=vae_pass(loader(m_train,shuffle=True,seed=SEED+epoch),vae_optimizer)
    validation=vae_pass(loader(m_val))
    vae_history.append({"epoch":epoch,**{"train_"+k:v for k,v in training.items()},**{"val_"+k:v for k,v in validation.items()}})
    if validation["objective"]<vae_best_loss:
        vae_best_loss=validation["objective"];vae_best_epoch=epoch
        vae_state={k:v.detach().cpu().clone() for k,v in vae.state_dict().items()}
    prior_frame();print(f"VAE epoch {epoch}: validation {validation}")
vae.load_state_dict(vae_state);vae.eval()
vae_test=vae_pass(loader(m_test),seed=SEED+201)
assert abs(vae_test["objective"]-(vae_test["reconstruction"]+BETA*vae_test["KL"]))<1e-3
frames[0].save(OUTPUT_DIR/"prior_training.gif",save_all=True,append_images=frames[1:],duration=700,loop=0)
with Image.open(OUTPUT_DIR/"prior_training.gif") as animation:assert animation.n_frames==len(frames)
vae_history=pd.DataFrame(vae_history)
fig,axes=plt.subplots(1,3,figsize=(13,3.3),layout="constrained")
for ax,metric in zip(axes,["objective","reconstruction","KL"]):
    ax.plot(vae_history.epoch,vae_history["train_"+metric],label="Training");ax.plot(vae_history.epoch,vae_history["val_"+metric],label="Validation")
    ax.axvline(vae_best_epoch,color="gray",ls="--");ax.set(title=metric,xlabel="Epoch",ylabel="Nats per image");ax.legend(fontsize=8)
save_figure(fig,"vae_curves.png")
print("Validation-selected epoch:",vae_best_epoch,"Test:",vae_test)

## 6. Reconstruction, prior generation, and Bernoulli samples

Posterior-mean reconstruction uses an input image. Prior generation uses `z ~ N(0,I)` and no image. The decoder sigmoid gives pixel probabilities; sampling each Bernoulli pixel produces a binary observation and adds visible noise. A smooth probability image is not a binary sample. The GIF follows the **same prior codes** before training and after each epoch, so visual changes are not caused by changing codes. Its last frame is the final epoch, which need not be the validation-selected epoch.

In [ ]:
with torch.no_grad():
    posterior_mu,posterior_logvar=vae.encode(m_test[:8].to(device))
    mean_reconstruction=vae.decode(posterior_mu).sigmoid().cpu()
    prior_probabilities=vae.decode(fixed_prior.to(device)).sigmoid().cpu()
    prior_binary=torch.bernoulli(prior_probabilities,generator=torch.Generator().manual_seed(SEED+301))
assert ((prior_probabilities>=0)&(prior_probabilities<=1)).all() and set(prior_binary.unique().tolist())<={0.0,1.0}
image_rows([m_test[:8],mean_reconstruction,prior_probabilities[:8],prior_binary[:8]],
    ["Input","Decode mu","Prior p(pixel)","Prior binary"],"vae_reconstruction_generation.png")
display(DisplayImage(filename=str(OUTPUT_DIR/"prior_training.gif")))

## 7. Explore the two-dimensional latent space

Plot held-out posterior means colored by digit labels (labels were not used to fit the VAE). Then decode a grid with z1 left→right and z2 bottom→top, and interpolate linearly between the first two held-out posterior means. The grid extends to ±2 standard deviations on each axis; its points are **not equally likely** under the Gaussian prior. Nearby codes need not form clean digit classes, and a smooth interpolation does not establish disentanglement or broad generative quality.

In [ ]:
with torch.no_grad():
    latent=torch.cat([vae.encode(x.to(device))[0].cpu() for x,_ in loader(m_test)])
    axis=torch.linspace(-2,2,10)
    grid_codes=torch.tensor([[x,y] for y in axis.flip(0).tolist() for x in axis.tolist()])
    grid_images=vae.decode(grid_codes.to(device)).sigmoid().cpu()
    alpha=torch.linspace(0,1,8)[:,None]
    interpolation=(1-alpha)*latent[0]+alpha*latent[1]
    interpolation_images=vae.decode(interpolation.to(device)).sigmoid().cpu()
fig,axes=plt.subplots(1,2,figsize=(10,4.5),layout="constrained")
scatter=axes[0].scatter(latent[:,0],latent[:,1],c=m_labels.numpy(),cmap="tab10",vmin=-0.5,vmax=9.5,s=8,alpha=0.65)
fig.colorbar(scatter,ax=axes[0],ticks=range(10),label="Digit (visualization only)")
axes[0].set(title="Held-out posterior means",xlabel="mu1",ylabel="mu2")
mosaic=grid_images[:,0].reshape(10,10,28,28).permute(0,2,1,3).reshape(280,280)
axes[1].imshow(mosaic,cmap="gray",vmin=0,vmax=1,extent=(-2.2,2.2,-2.2,2.2))
axes[1].set(title="Decode a latent grid",xlabel="z1",ylabel="z2",xticks=[-2,0,2],yticks=[-2,0,2])
save_figure(fig,"vae_latent_space.png")
image_rows([interpolation_images],["mu interpolation"],"vae_interpolation.png")

## 8. Advanced: adversarial autoencoder (AAE)

This small default-enabled demonstration keeps **one shared encoder**. Reconstruction updates that encoder and decoder. Discriminator training uses detached encoded data versus N(0,I), labeled 0 and 1. The adversarial encoder step then targets 1 through a frozen discriminator. Freezing discriminator parameters still allows gradients through its input.

An AAE targets the **aggregate code distribution**, unlike the VAE's per-example KL. These objectives are not interchangeable. We use a single encoder optimizer across its two update steps; no second unrelated encoder is created. Snapshot assertions verify which networks change during each phase. Two quick-mode epochs check the mechanism, not successful prior matching or sharp generation. Select the saved AAE on validation reconstruction loss; this criterion does not select generative quality.

In [ ]:
aae_report={"executed":False}
if RUN_AAE:
    class AAE(nn.Module):
        def __init__(self):
            super().__init__()
            self.encoder=nn.Sequential(nn.Flatten(),nn.Linear(784,128),nn.ReLU(),nn.Linear(128,2))
            self.decoder=nn.Sequential(nn.Linear(2,128),nn.ReLU(),nn.Linear(128,784),nn.Unflatten(1,(1,28,28)))
            self.discriminator=nn.Sequential(nn.Linear(2,32),nn.ReLU(),nn.Linear(32,1))
        def forward(self,x):return self.decoder(self.encoder(x))
    aae=AAE().to(device)
    enc_opt=torch.optim.Adam(aae.encoder.parameters(),lr=0.001)
    dec_opt=torch.optim.Adam(aae.decoder.parameters(),lr=0.001)
    disc_opt=torch.optim.Adam(aae.discriminator.parameters(),lr=0.001)
    def snapshot(module):return [p.detach().clone() for p in module.parameters()]
    def same(module,state):return all(torch.equal(p.detach(),q) for p,q in zip(module.parameters(),state))
    aae_epochs=2 if QUICK_RUN else 10;aae_train=m_train[:2000] if QUICK_RUN else m_train
    aae_history=[];aae_best=float("inf");audit_done=False
    for epoch in range(1,aae_epochs+1):
        sums=np.zeros(3);number=0;aae.train()
        for x,_ in loader(aae_train,shuffle=True,seed=SEED+epoch):
            x=x.to(device)
            if not audit_done:enc_before=snapshot(aae.encoder);disc_before=snapshot(aae.discriminator)
            # Reconstruction: shared encoder + decoder only.
            enc_opt.zero_grad(set_to_none=True);dec_opt.zero_grad(set_to_none=True)
            recon=F.binary_cross_entropy_with_logits(aae(x),x);assert torch.isfinite(recon)
            recon.backward();enc_opt.step();dec_opt.step()
            if not audit_done:
                assert not same(aae.encoder,enc_before) and same(aae.discriminator,disc_before)
                enc_before=snapshot(aae.encoder);dec_before=snapshot(aae.decoder)
            # Discriminator: detached codes, so encoder receives no new gradients.
            enc_opt.zero_grad(set_to_none=True);disc_opt.zero_grad(set_to_none=True)
            encoded=aae.encoder(x).detach();prior=torch.randn_like(encoded)
            dloss=0.5*(F.binary_cross_entropy_with_logits(aae.discriminator(prior),torch.ones(len(x),1,device=device))+
                       F.binary_cross_entropy_with_logits(aae.discriminator(encoded),torch.zeros(len(x),1,device=device)))
            assert torch.isfinite(dloss);dloss.backward();disc_opt.step()
            if not audit_done:
                assert same(aae.encoder,enc_before) and same(aae.decoder,dec_before)
                assert all(p.grad is None for p in aae.encoder.parameters())
                disc_before=snapshot(aae.discriminator)
            # Adversarial encoder: same encoder; frozen discriminator and untouched decoder.
            for p in aae.discriminator.parameters():p.requires_grad_(False)
            enc_opt.zero_grad(set_to_none=True)
            adversarial=F.binary_cross_entropy_with_logits(aae.discriminator(aae.encoder(x)),torch.ones(len(x),1,device=device))
            assert torch.isfinite(adversarial);adversarial.backward();enc_opt.step()
            if not audit_done:
                assert not same(aae.encoder,enc_before) and same(aae.discriminator,disc_before) and same(aae.decoder,dec_before)
                audit_done=True
            for p in aae.discriminator.parameters():p.requires_grad_(True)
            sums+=np.array([recon.item(),dloss.item(),adversarial.item()])*len(x);number+=len(x)
        aae.eval();val_sum=val_count=0
        with torch.no_grad():
            for x,_ in loader(m_val):
                x=x.to(device);val_sum+=F.binary_cross_entropy_with_logits(aae(x),x,reduction="sum").item();val_count+=x.numel()
        val_loss=val_sum/val_count
        aae_history.append({"epoch":epoch,"reconstruction":sums[0]/number,"discriminator":sums[1]/number,"adversarial":sums[2]/number,"val_reconstruction":val_loss})
        if val_loss<aae_best:
            aae_best=val_loss;aae_best_epoch=epoch;aae_state={k:v.detach().cpu().clone() for k,v in aae.state_dict().items()}
        print("AAE:",aae_history[-1])
    aae.load_state_dict(aae_state);aae.eval()
    with torch.no_grad():aae_prior=aae.decoder(fixed_prior.to(device)).sigmoid().cpu()
    image_rows([aae_prior[:8]],["AAE prior"],"aae_prior.png")
    aae_report={"executed":True,"epochs":aae_epochs,"training_images":len(aae_train),"best_epoch":aae_best_epoch,
                "history":aae_history,"shared_encoder_and_update_isolation":audit_done,
                "interpretation":"Short mechanism check; validation reconstruction does not measure generative quality."}

## 9. Save models, reload on CPU, and export evidence

Store weights together with preprocessing, architecture settings, noise scale, and the ECG calibration threshold. Compare deterministic outputs after CPU reload. For the VAE, supply the **same epsilon** to both models; independently sampled reconstructions should not be expected to match. Fixed prior codes also verify decoder equality. The model classes above are needed to reconstruct these checkpoints.

In [ ]:
models_to_save={"dense_ae":(ae,DenseAE,f_test[:4]),"denoiser":(denoiser,Denoiser,noisy_test[:4]),
                "ecg_ae":(ecg_model,ECGAutoencoder,ecg_test[:4])}
if RUN_AAE:models_to_save["aae"]=(aae,AAE,m_test[:4])
reload_checks={}
for name,(network,constructor,probe) in models_to_save.items():
    network.eval()
    payload={"state_dict":{k:v.detach().cpu() for k,v in network.state_dict().items()},"model_class":constructor.__name__,
             "pixel_scaling":"Fashion /255; MNIST /255 then >0.5","noise_sigma":0.3,
             "ecg_scaling":{"min":low,"range":scale},"ecg_threshold":threshold,"ecg_positive_label":0}
    torch.save(payload,OUTPUT_DIR/f"{name}.pt")
    saved=torch.load(OUTPUT_DIR/f"{name}.pt",map_location="cpu",weights_only=True)
    restored=constructor().eval();restored.load_state_dict(saved["state_dict"])
    with torch.no_grad():torch.testing.assert_close(network(probe.to(device)).cpu(),restored(probe),rtol=1e-4,atol=1e-5)
    if name=="ecg_ae":assert saved["ecg_threshold"]==threshold and saved["ecg_positive_label"]==0
    reload_checks[name]=True
vae_payload={"state_dict":{k:v.detach().cpu() for k,v in vae.state_dict().items()},"latent_dim":LATENT_DIM,
             "beta":BETA,"binarization_threshold":0.5,"best_epoch":vae_best_epoch}
torch.save(vae_payload,OUTPUT_DIR/"vae.pt")
saved=torch.load(OUTPUT_DIR/"vae.pt",map_location="cpu",weights_only=True)
restored_vae=ConvVAE(saved["latent_dim"]).eval();restored_vae.load_state_dict(saved["state_dict"])
epsilon=torch.randn(4,LATENT_DIM,generator=torch.Generator().manual_seed(SEED+500))
with torch.no_grad():
    original=vae(m_test[:4].to(device),epsilon.to(device))[0].cpu()
    replayed=restored_vae(m_test[:4],epsilon)[0]
    torch.testing.assert_close(original,replayed,rtol=1e-4,atol=1e-5)
    torch.testing.assert_close(vae.decode(fixed_prior.to(device)).cpu(),restored_vae.decode(fixed_prior),rtol=1e-4,atol=1e-5)
reload_checks["vae"]=True
pd.DataFrame({"true_anomaly":y_anomaly,"predicted_anomaly":predicted_anomaly,"MAE":test_errors}).to_csv(OUTPUT_DIR/"ecg_predictions.csv",index=False)
assert len(pd.read_csv(OUTPUT_DIR/"ecg_predictions.csv"))==len(y_anomaly)
report={"environment":versions,"fashion_revision":FASHION_REVISION,
    "fashion_splits":[len(x) for x in (f_train,f_val,f_test)],"mnist_splits":[len(x) for x in (m_train,m_val,m_test)],
    "image_metrics":image_metrics,"image_best_epochs":{"ae":ae_best,"denoiser":dn_best},
    "ecg":{"url":ECG_URL,"sha256":ecg_sha,"rows":len(df),"duplicates_removed":before-len(df),
           "splits":[len(i) for i in (train_idx,val_idx,test_idx)],"normal_training":len(ecg_train),"normal_calibration":len(ecg_val),
           "best_epoch":ecg_best,"metrics":ecg_metrics},
    "vae":{"latent_dim":LATENT_DIM,"beta":BETA,"epochs":vae_epochs,"best_epoch":vae_best_epoch,"test":vae_test,"gif_frames":len(frames)},
    "aae":aae_report,"checkpoint_reload":reload_checks,"figures":figures,
    "checks":{"split_indices":True,"binary_mnist":True,"analytic_KL":True,"reparameterization_gradients":True,
              "finite_losses":True,"anomaly_label_direction":True,"vae_objective_units":True,"gif_roundtrip":True},
    "seconds_after_imports":time.perf_counter()-run_started}
(OUTPUT_DIR/"metrics.json").write_text(json.dumps(report,indent=2,allow_nan=False))
print("TUTORIAL_10_COMPLETE");print(json.dumps(report,indent=2,allow_nan=False))

## Exercises

1. Compare AE reconstruction with the training-mean image. Why does good reconstruction not guarantee useful random latent samples?
2. Change training noise strength and use a prespecified validation corruption. Does denoising improve over the unchanged input?
3. Explain the ECG positive label and the difference between calibration exceedance and held-out false-positive rate. Why is the split not patient independent?
4. Change beta using validation results. Keep reconstruction and KL units consistent. For beta≠1, rename the objective rather than calling it the standard negative ELBO.
5. Compare decoding mu, sampling from q(z|x), and sampling from p(z). Identify where each random variable enters.
6. Explain why training a second encoder in the AAE would not regularize the decoder's actual input distribution.

Download `outputs/tutorial_10/` before disconnecting. Larger mode uses 15 reconstruction epochs, 150 ECG epochs, 20 VAE epochs, and 10 AAE epochs. GPU/larger mode and the AAE-skip branch require separate validation. Small images, limited training, and row-level ECG splits constrain what these results establish.

## Troubleshooting and next steps

- Run cells from the top after a runtime restart; a saved notebook does not preserve Python variables.
- If a data download fails, check connectivity and rerun that cell. If imports fail, use a fresh runtime.
- Download files from `outputs/tutorial_10/` using Colab’s Files sidebar before disconnecting.
- Change one experimental setting at a time and record what you changed.

## Attribution

License information is available in the repository `LICENSE`.

References: [Kingma & Welling, Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114), [Makhzani et al., Adversarial Autoencoders](https://arxiv.org/abs/1511.05644), [Fashion-MNIST](https://github.com/zalandoresearch/fashion-mnist), [PyTorch MNIST](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html), and the [TensorFlow ECG/autoencoder data description](https://www.tensorflow.org/tutorials/generative/autoencoder). The ECG download is a data file; this notebook does not import TensorFlow. Data terms are separate from the MIT code license. No data or trained weights are committed.

This adaptation fixes forced-CUDA/device mismatches, test-set model selection, reversed anomaly labels, stochastic-evaluation ambiguity, and the source AAE's separate-encoder bug. It distinguishes posterior reconstruction from prior generation and records model reload checks.